In [ ]:


import os
import random
import numpy as np
import tensorflow as tf
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../..")))

SEED = 1234

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

from project.loader import load
from project.CROP.models.crop2 import Crop2 

dataset = "mnist"

cvae = load.cvae(inter=512,lat=64,dataset=dataset)
data = load.data(dataset=dataset)
predictor = load.predictor(dataset=dataset)

x_test = data["x_test"]
x_test_1 = data["x_test_1"]
y_test = data["y_test"]
y_test_1 = data["y_test_1"]


c = Crop2(cvae=cvae,predictor=predictor)

def objective(trial):

    bias = trial.suggest_float("bias", 0.17,0.19)
    slope = trial.suggest_float("slope", 11,14 )
    gamma = trial.suggest_float("gamma", 0.25, 0.37)

    n_images=2000
    try:        
        metrics = c.unmix(
            x_test[:n_images],
            x_test_1[:n_images],
            y_test[:n_images],
            y_test_1[:n_images],
            params={"gamma": gamma,"bias":bias,"slope":slope},
            iterations=10,
            show_metrics=False
        )
        met = float(metrics["acc_both"])
        return  met

    except Exception as e:
        print(f"Error con bias={bias}, slope={slope}: {e}")
        return float("inf")
    



2026-01-20 20:21:17.049791: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-20 20:21:17.050190: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-20 20:21:17.052801: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-20 20:21:17.059793: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768951277.071199 2295855 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768951277.07

Usando mnist como dataset


In [14]:
import optuna


# Creamos un estudio de minimización
study = optuna.create_study(direction = "maximize")
study.optimize(objective, n_trials=200) # Probamos 30 combinaciones

# Mostramos los mejores resultados
print("\n📊 Mejores hiperparámetros encontrados:")
print(study.best_trials)

#params={'bias': 0.19730951040100445, 'slope': 12.186034315346337, 'gamma': 0.376136883819422}
#params={'bias': 0.18187551159723622, 'slope': 12.988917761056788, 'gamma': 0.3330581352295742}
#params={'bias': 0.17850015445986958, 'slope': 12.927274351283915, 'gamma': 0.2522317775074754}

[I 2026-01-14 00:39:44,284] A new study created in memory with name: no-name-cb4a8696-1ba7-4d64-9b98-6440285f6f6e
[I 2026-01-14 00:39:46,883] Trial 0 finished with value: 0.811 and parameters: {'bias': 0.18321950252620556, 'slope': 12.855411564804825, 'gamma': 0.3340821056924871}. Best is trial 0 with value: 0.811.
[I 2026-01-14 00:39:49,544] Trial 1 finished with value: 0.812 and parameters: {'bias': 0.17309960139589256, 'slope': 13.952289040735117, 'gamma': 0.36365593806826463}. Best is trial 1 with value: 0.812.
[I 2026-01-14 00:39:51,928] Trial 2 finished with value: 0.8085 and parameters: {'bias': 0.17910611726274084, 'slope': 12.966919195463426, 'gamma': 0.2929733941402611}. Best is trial 1 with value: 0.812.
[I 2026-01-14 00:39:54,684] Trial 3 finished with value: 0.81 and parameters: {'bias': 0.1804283678466045, 'slope': 11.908496201705297, 'gamma': 0.2928302830199859}. Best is trial 1 with value: 0.812.
[I 2026-01-14 00:39:57,244] Trial 4 finished with value: 0.809 and paramet


📊 Mejores hiperparámetros encontrados:
[FrozenTrial(number=116, state=<TrialState.COMPLETE: 1>, values=[0.818], datetime_start=datetime.datetime(2026, 1, 14, 0, 44, 50, 231430), datetime_complete=datetime.datetime(2026, 1, 14, 0, 44, 52, 732569), params={'bias': 0.17674774880353092, 'slope': 13.98070127463459, 'gamma': 0.3677650462648937}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'bias': FloatDistribution(high=0.19, log=False, low=0.17, step=None), 'slope': FloatDistribution(high=14.0, log=False, low=11.0, step=None), 'gamma': FloatDistribution(high=0.37, log=False, low=0.25, step=None)}, trial_id=116, value=None)]


In [ ]:
n_image=10
labels=["0","1","2","3","4","5","6","7","8","9","0"]


#parámetros optimizados para ssim
ssim_params={'bias': 0.3490758398711943, 'slope': 23.79981809771706, 'alpha_1': -2.5004348572265864, 'alpha_2': -16.418488717733673, 'gamma': 0.2612293606781744}
#parámetros optimizados para recon_bpsnr
recon_bpsnr_params={'bias': 0.1753106428802752, 'slope': 20.088006577214188, 'alpha_1': -4.3120902863188775, 'alpha_2': -31.469765177549906, 'gamma': 0.7367839585631266}
#parámetros optimizados para mask_bpsnr
mask_bpsnr_params={'bias': 0.4032196468954787, 'slope': 10.773311686035228, 'alpha_1': -4.487552303478556, 'alpha_2': -7.255734630588293, 'gamma': 0.44933267965195817}#parámetros optimizados para acc_both
#parámetros optimizados para acc_both
acc_both_params={'bias': 0.21716868437244724, 'slope': 19.437646607877017, 'alpha_1': -0.13863158439804657, 'alpha_2': -10.463749682840565, 'gamma': 0.5392507616026618}
## solo optimizanddo gamma
acc_both_gamma = {'gamma': 0.7435064454594396}

##recon_bpsnr lat128_inter_512
params_recon_bpsnr_lat128_inter_512={'gamma': 0.9519615639307367}#optimizacion para recon_bpsnr
params_acc_both_lat128_inter_512 = {'gamma': 0.3012911033494221} # optimización de acc
#acc_both bias y slope y gamma 

results = c.unmix(
    x_test[:n_image],
    x_test_1[:n_image],
    y_test[:n_image],
    y_test_1[:n_image],
    #params={'bias': 0.17786223110878405, 'slope': 24.277471484785064, 'gamma': 0.4665746345952185},
    params = recon_bpsnr_params,
    iterations=10,
    show_image=True,
    show_metrics=False,
    labels=labels
)